# 02 — Relajación Lagrangiana: desarrollo y validación

Notebook de desarrollo incremental del algoritmo de Relajación Lagrangiana
para el modelo HDM (Li et al., 2026), adaptado al caso de estudio de
San Cristóbal de La Laguna.

**Pipeline del notebook:**
1. Validación de subproblemas (P_LR_x, P_LR_z)
2. Validación del repair (solución factible)
3. Bucle completo del subgradiente
4. Comparación con modelo exacto (Gurobi)

In [3]:
import sys
sys.path.insert(0, "../src/python")

import numpy as np
import warnings
from instancia import load_instance
from lagrangiana import (
    solve_plr_x, solve_plr_x_greedy,
    repair_solution, precompute_valid_candidates,
    Multipliers, compute_demand,
)

inst = load_instance("../data/processed/instancia_laguna.json")
n_j, n_k = len(inst.J), len(inst.K)

print(f"Instancia: {inst.study_case}")
print(f"  Edificios (I):   {len(inst.I)}")
print(f"  Candidatos (J):  {n_j}")
print(f"  Tipos residuo:   {n_k}")

Instancia: San Cristobal de La Laguna - Plaza del Cristo
  Edificios (I):   740
  Candidatos (J):  133
  Tipos residuo:   4


## 1. Validación de subproblemas

### 1.1 P_LR_x — Greedy vs Gurobi

Comparamos el subproblema de bins resuelto con greedy (paper original)
y con Gurobi (nuestra mejora). La diferencia clave es la desigualdad
válida Σₖ x[j,k] ≤ N_j que Gurobi puede imponer.

In [4]:
mults_zero = Multipliers(
    mu=np.zeros((n_j, n_k)),
    lbd=np.zeros(n_j),
    nu=np.zeros((n_j, n_k)),
)

x_g, obj_g = solve_plr_x_greedy(inst, mults_zero)
x_gb, obj_gb = solve_plr_x(inst, mults_zero)

print("Test 1 — Multiplicadores = 0 (iteración 0)")
print(f"  Greedy:  obj = {obj_g:>12,.2f}   bins = {x_g.sum()}")
print(f"  Gurobi:  obj = {obj_gb:>12,.2f}   bins = {x_gb.sum()}")
print(f"  Δ obj:         {obj_gb - obj_g:>12,.2f}")
print(f"  Max bins/punto — greedy: {x_g.sum(axis=1).max()}  gurobi: {x_gb.sum(axis=1).max()}")

Test 1 — Multiplicadores = 0 (iteración 0)
  Greedy:  obj =    34,100.00   bins = 109
  Gurobi:  obj =    34,100.00   bins = 109
  Δ obj:                 0.00
  Max bins/punto — greedy: 28  gurobi: 8


In [5]:
rng = np.random.default_rng(42)
mults_avanzados = Multipliers(
    mu=rng.uniform(0, 5, size=(n_j, n_k)),
    lbd=rng.uniform(0, 50, size=n_j),
    nu=rng.uniform(0, 10, size=(n_j, n_k)),
)

x_g2, obj_g2 = solve_plr_x_greedy(inst, mults_avanzados)
x_gb2, obj_gb2 = solve_plr_x(inst, mults_avanzados)

bin_cost_arr = np.array([inst.params.bin_cost[k] for k in inst.K])
bin_cap_arr = np.array([inst.params.bin_capacity[k] for k in inst.K])
coef = (bin_cost_arr + mults_avanzados.lbd[:, np.newaxis]
        + mults_avanzados.nu - mults_avanzados.mu * bin_cap_arr)

print("Test 2 — Multiplicadores aleatorios (simula iteración avanzada)")
print(f"  Coeficientes negativos: {(coef < 0).sum()} de {coef.size}")
print(f"  Greedy:  obj = {obj_g2:>12,.2f}   bins = {x_g2.sum()}")
print(f"  Gurobi:  obj = {obj_gb2:>12,.2f}   bins = {x_gb2.sum()}")
print(f"  Δ obj:         {obj_gb2 - obj_g2:>12,.2f}")
print(f"  Max bins/punto — greedy: {x_g2.sum(axis=1).max()}  gurobi: {x_gb2.sum(axis=1).max()}")

Test 2 — Multiplicadores aleatorios (simula iteración avanzada)
  Coeficientes negativos: 193 de 532
  Greedy:  obj =  -196,730.68   bins = 1544
  Gurobi:  obj =  -143,231.42   bins = 920
  Δ obj:            53,499.27
  Max bins/punto — greedy: 24  gurobi: 8


## 2. Validación del repair

Construimos una z de prueba (abriendo los 25 candidatos con mayor
cobertura) y verificamos que `repair_solution` produce una solución
factible: asignación completa, bins correctos, y sin violaciones de N_j.

In [6]:
vc = precompute_valid_candidates(inst)
print(f"valid_candidates: {len(vc)} edificios × {len(vc[0])} tipos")

# Construir z de prueba: abrir los 25 candidatos con mayor cobertura
coverage_count = np.zeros(n_j, dtype=int)
for i_idx in range(len(inst.I)):
    for k in range(n_k):
        for j_idx in vc[i_idx][k]:
            coverage_count[j_idx] += 1

z_test = np.zeros(n_j, dtype=int)
top_25 = np.argsort(coverage_count)[-25:]
z_test[top_25] = 1
print(f"Puntos abiertos: {z_test.sum()}")

valid_candidates: 740 edificios × 4 tipos
Puntos abiertos: 25


In [7]:
sol = repair_solution(inst, z_test, vc)

print(f"Resultado del repair:")
print(f"  Puntos abiertos (tras repair): {sol.z.sum()}")
print(f"  Bins totales:                  {sol.x.sum()}")
print(f"  Coste (upper bound):           {sol.cost:,.1f} €")
print(f"  Max bins en un punto:          {sol.x.sum(axis=1).max()}")
print(f"  Edificios sin asignar:         {(sol.y_assign == -1).sum()}")

UnboundLocalError: cannot access local variable 'z_rep' where it is not associated with a value

## 3. Ejecución completa — Bucle del subgradiente

Primera ejecución con 200 iteraciones para verificar convergencia.
Comparamos después con el modelo exacto (156.360€ - Solución Óptima).

In [2]:
from lagrangiana import solve_lagrangian
import time

print("=" * 65)
print("  Relajación Lagrangiana — HDM")
print("=" * 65)
print(f"  Instancia: {inst.study_case}")
print(f"  I={len(inst.I)}  J={len(inst.J)}  K={len(inst.K)}")
print(f"  Referencia: Gurobi óptimo = 156,350 € (gap 0%)")
print("-" * 65)

start = time.time()
result = solve_lagrangian(
    inst,
    max_iterations=200,
    gap_tolerance=0.01,
    no_improve_limit=50,
    verbose=True,
    print_every=10,
)
elapsed = time.time() - start

print("-" * 65)
print(f"  Iteraciones:     {result.n_iterations}")
print(f"  Tiempo:          {elapsed:.1f}s")
print(f"  Best LB:         {result.best_lb:>12,.1f} €")
print(f"  Best UB:         {result.best_ub:>12,.1f} €")
print(f"  Gap LR:          {result.gap:.2%}")
print(f"  Puntos abiertos: {int(np.sum(result.best_feasible.z))}")
print(f"  Bins totales:    {int(np.sum(result.best_feasible.x))}")
print(f"  Óptimo Gurobi:   156,350 €")
print(f"  Dist. al óptimo: {result.best_ub - 156_350:+,.1f} € ({(result.best_ub - 156_350)/156_350:.2%})")
print("=" * 65)

  Relajación Lagrangiana — HDM
  Instancia: San Cristobal de La Laguna - Plaza del Cristo
  I=740  J=133  K=4
  Referencia: Gurobi óptimo = 156,350 € (gap 0%)
-----------------------------------------------------------------
Restricted license - for non-production use only - expires 2027-11-29


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

iters = range(len(result.lb_history))
ax.plot(iters, result.lb_history, label="Lower Bound (LB)", linewidth=2)
ax.plot(iters, result.ub_history, label="Upper Bound (UB)", linewidth=2)
ax.axhline(y=156_350, color="red", linestyle="--", alpha=0.7, label="Óptimo Gurobi (156.350 €)")

ax.set_xlabel("Iteración")
ax.set_ylabel("Coste (€)")
ax.set_title("Convergencia de la Relajación Lagrangiana")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Pruebas

In [ ]:
"""
02 — Relajación Lagrangiana: desarrollo y validación
Validación de subproblemas, convergencia y comparación con modelo exacto.
"""
import sys
sys.path.insert(0, "../src/python")

import numpy as np
from instancia import load_instance
from lagrangiana import (
    solve_plr_x, solve_plr_x_greedy,
    Multipliers, compute_demand
)

inst = load_instance("../data/processed/instancia_laguna.json")
print(f"Instancia: {inst.study_case}")
print(f"  I={len(inst.I)}  J={len(inst.J)}  K={len(inst.K)}")

In [ ]:
n_j, n_k = len(inst.J), len(inst.K)

mults = Multipliers(
    mu=np.zeros((n_j, n_k)),
    lbd=np.zeros(n_j),
    nu=np.zeros((n_j, n_k)),
)

x_g, obj_g = solve_plr_x_greedy(inst, mults)
x_gurobi, obj_gurobi = solve_plr_x(inst, mults)

print(f"Greedy:  obj = {obj_g:>12,.2f}   bins = {x_g.sum()}")
print(f"Gurobi:  obj = {obj_gurobi:>12,.2f}   bins = {x_gurobi.sum()}")
print(f"Δ obj:         {obj_gurobi - obj_g:>12,.2f}")
print(f"Max bins/punto — greedy: {x_g.sum(axis=1).max()}  gurobi: {x_gurobi.sum(axis=1).max()}")